In [2]:
# Cell 1 — set working directory to project root
import os
os.chdir('..')
print(os.getcwd())  # should print: .../olist-ops-analysis

/Users/ariellacallista/Downloads/olist-ops-analysis


# Olist Brazilian E-Commerce — Operations Analysis

## Project context
Olist is a Brazilian e-commerce marketplace that connects small sellers to 
major retail channels. This notebook analyses 100k+ orders to identify 
operational bottlenecks in the fulfillment pipeline.

## Business questions
1. What is the overall on-time delivery rate, and how does it vary by state?
2. What is the average seller processing time, and who are the slowest sellers?
3. How does delivery delay correlate with customer review score?
4. Which product categories have the highest late delivery rates?
5. What does the monthly order volume trend look like — are there seasonal patterns?

## Tools used
- Python (pandas, numpy) — data cleaning and feature engineering
- BigQuery SQL — analytical queries
- Power BI — dashboard and visualisation

## Phase 1 — Data exploration

### Key findings from profiling

**orders table (99,441 rows, 8 columns)**
- All date columns stored as strings — need to be parsed to datetime
- Nulls in date columns are expected: orders that were never approved,
  shipped, or delivered will naturally have empty date fields
- order_approved_at: 160 nulls
- order_delivered_carrier_date: 1,783 nulls
- order_delivered_customer_date: 2,965 nulls

**order_status breakdown**
- delivered: 96,478 (97% of all orders)
- shipped: 1,107
- canceled: 625
- unavailable: 609
- Other statuses: <1% each

**order_items table (112,650 rows)**
- More rows than orders because some orders contain multiple items
- order_item_id indicates item position within an order (1, 2, 3...)
- 88,863 orders have 1 item (~79%), remainder have 2 or more
- No nulls in any column

**customers table (99,441 rows)**
- Clean — no nulls in any column
- São Paulo (SP) dominates with 41,746 orders (~42%)
- Top 3 states: SP, RJ, MG account for ~67% of all orders

**products table (32,951 rows)**
- 610 nulls in product_category_name and dimension columns
- product_weight_g and dimension columns have 2 nulls each

**reviews table (99,224 rows)**
- 87,656 nulls in review_comment_title (most customers only leave a score)
- 58,247 nulls in review_comment_message
- 551 orders have duplicate reviews (multiple submissions)

**product_category_name_translation (71 rows)**
- 623 products in the products table have no English translation

In [3]:
import pandas as pd
import os

path = "data/raw/"
files = [f for f in os.listdir(path) if f.endswith('.csv')]

for f in files:
    df = pd.read_csv(os.path.join(path, f))
    print(f"\n--- {f} ---")
    print(f"Shape: {df.shape}")
    print(df.dtypes)
    print(df.isnull().sum())


--- olist_sellers_dataset.csv ---
Shape: (3095, 4)
seller_id                 object
seller_zip_code_prefix     int64
seller_city               object
seller_state              object
dtype: object
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

--- product_category_name_translation.csv ---
Shape: (71, 2)
product_category_name            object
product_category_name_english    object
dtype: object
product_category_name            0
product_category_name_english    0
dtype: int64

--- olist_orders_dataset.csv ---
Shape: (99441, 8)
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object
order_id                            0
customer_id   

## Phase 2 - Data Cleaning

In [4]:
import os

files = [f for f in os.listdir("data/raw/") if f.endswith('.csv')]
for f in files:
    print(f)

olist_sellers_dataset.csv
product_category_name_translation.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv


In [5]:
# Load all tables
import pandas as pd
import os

path = "data/raw/"

orders = pd.read_csv(path + "olist_orders_dataset.csv")
order_items = pd.read_csv(path + "olist_order_items_dataset.csv")
customers = pd.read_csv(path + "olist_customers_dataset.csv")
sellers = pd.read_csv(path + "olist_sellers_dataset.csv")
products = pd.read_csv(path + "olist_products_dataset.csv")
payments = pd.read_csv(path + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(path + "olist_order_reviews_dataset.csv")
geolocation = pd.read_csv(path + "olist_geolocation_dataset.csv")
translation = pd.read_csv(path + "product_category_name_translation.csv")

print("All tables loaded.")


All tables loaded.


### orders table
1. Parsed all 5 date columns from string to datetime using pd.to_datetime()
   with errors='coerce' to handle malformed values gracefully
2. Filtered to delivered orders only (96,478 rows) for delivery analysis
   Kept orders_all as a separate copy for cancellation rate analysis
3. Engineered 5 operational metrics at row level:
   - processing_days: order_approved_at minus order_purchase_timestamp
   - shipping_days: order_delivered_carrier_date minus order_approved_at
   - delivery_days: order_delivered_customer_date minus order_purchase_timestamp
   - delay_days: order_delivered_customer_date minus order_estimated_delivery_date
     (negative = arrived early, positive = arrived late)
   - is_late: binary flag, 1 if delay_days > 0
4. Cleaned outliers:
   - 1,350 negative shipping_days nulled out — caused by batch midnight
     approval timestamps being recorded after carrier pickup (logging error)
   - delivery_days > 60 nulled out (306 rows) — beyond reasonable analysis range
   - shipping_days > 30 nulled out (177 rows)
   - processing_days > 10 nulled out (38 rows)
5. Added time columns for trend analysis:
   - order_month (YYYY-MM format)
   - order_year
   - order_quarter

In [6]:
# Profile the orders table
print(orders.shape)
print(orders.dtypes)
print(orders.isnull().sum())

(99441, 8)
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [7]:
# Fix date columns -- ORDERS
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

print(orders.dtypes)
print(orders.isnull().sum())

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [8]:
# Check order status breakdown -- need to see the status breakdown before we filter by delivered only
print(orders['order_status'].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [9]:
# Setp 5 -- Filter to delivered orders

# keep a copy of all orders for cancellation analysis later
orders_all = orders.copy()

# filter to delivered only for delivery analysis
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

print(f"Total orders: {len(orders)}")
print(f"Delivered orders: {len(orders_delivered)}")
print(f"Dropped: {len(orders) - len(orders_delivered)}")


Total orders: 99441
Delivered orders: 96478
Dropped: 2963


In [10]:
# Step 6 -- Create operational metrics
# 1. processing days -> to compute avg processing days
orders_delivered['processing_days'] = (
    orders_delivered['order_approved_at'] - 
    orders_delivered['order_purchase_timestamp']
).dt.total_seconds() / 86400

# 2. shipping days
orders_delivered['shipping_days'] = (
    orders_delivered['order_delivered_carrier_date'] - 
    orders_delivered['order_approved_at']
).dt.total_seconds() / 86400

# 3. delivery days
orders_delivered['delivery_days'] = (
    orders_delivered['order_delivered_customer_date'] - 
    orders_delivered['order_purchase_timestamp']
).dt.total_seconds() / 86400

# 4. delay days -- negative means no delay
orders_delivered['delay_days'] = (
    orders_delivered['order_delivered_customer_date'] - 
    orders_delivered['order_estimated_delivery_date']
).dt.total_seconds() / 86400 

# 5. is late
orders_delivered['is_late'] = (orders_delivered['delay_days'] > 0).astype(int)

print(orders_delivered[['processing_days', 'shipping_days', 'delivery_days', 'delay_days', 'is_late']].describe().round(2))

       processing_days  shipping_days  delivery_days  delay_days   is_late
count         96464.00       96462.00       96470.00    96470.00  96478.00
mean              0.43           2.80          12.56      -11.18      0.08
std               0.86           3.54           9.55       10.18      0.27
min               0.00        -171.22           0.53     -146.02      0.00
25%               0.01           0.87           6.77      -16.24      0.00
50%               0.01           1.82          10.22      -11.95      0.00
75%               0.60           3.58          15.72       -6.39      0.00
max              30.89         125.76         209.63      188.98      1.00


In [11]:
# Step 7 -- inspect and remove impossible values

# check how many negative shipping days exist
print("Negative shipping days:", (orders_delivered['shipping_days'] < 0).sum())

# check how many extreme outliers exist
print("Delivery > 60 days:", (orders_delivered['delivery_days'] > 60).sum())
print("Shipping > 30 days:", (orders_delivered['shipping_days'] > 30).sum())
print("Processing > 10 days:", (orders_delivered['processing_days'] > 10).sum())

# look at worst offenders
print(orders_delivered[orders_delivered['shipping_days'] < 0][['order_id', 'order_purchase_timestamp', 
                                                               'order_approved_at', 'order_delivered_carrier_date', 
                                                               'shipping_days']].head(10))


Negative shipping days: 1350
Delivery > 60 days: 306
Shipping > 30 days: 177
Processing > 10 days: 38
                             order_id order_purchase_timestamp  \
15   dcb36b511fcac050b97cd5c05de84dc3      2018-06-07 19:03:12   
64   688052146432ef8253587b930b01a06d      2018-04-22 08:48:13   
199  58d4c4747ee059eeeb865b349b41f53a      2018-07-21 12:49:32   
210  412fccb2b44a99b36714bca3fef8ad7b      2018-07-22 22:30:05   
415  56a4ac10a4a8f2ba7693523bb439eede      2018-07-22 13:04:47   
481  32e4fa9bb468884309b58b9348de70c3      2018-07-04 16:49:21   
483  4df92d82d79c3b52c7138679fa9b07fc      2018-07-24 11:32:11   
585  16e38caa92e342c7780f68832f832d4d      2018-05-07 01:09:09   
615  b9afddbdcfadc9a87b41a83271c3e888      2018-08-16 13:50:48   
817  6051e6d3da9a50b7325cbe9c81025062      2018-07-03 23:40:16   

      order_approved_at order_delivered_carrier_date  shipping_days  
15  2018-06-12 23:31:02          2018-06-11 14:54:00      -1.359051  
64  2018-04-24 18:25:22        

In [12]:
# Step 8 - clean outliers

# null out negative shipping days -- logging error, order is still valid
orders_delivered.loc[orders_delivered['shipping_days'] < 0, 'shipping_days'] = None

# null out extreme outliers - beyond reasonable ops analysis range
orders_delivered.loc[orders_delivered['delivery_days'] > 60, 'delivery_days'] = None
orders_delivered.loc[orders_delivered['shipping_days'] > 30, 'shipping_days'] = None
orders_delivered.loc[orders_delivered['processing_days'] > 10, 'processing_days'] = None

# verify cleaned stats
print(orders_delivered[orders_delivered['shipping_days'] < 0][['order_id', 'order_purchase_timestamp', 
                                                               'order_approved_at', 'order_delivered_carrier_date', 
                                                               'shipping_days']].head(10))



Empty DataFrame
Columns: [order_id, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, shipping_days]
Index: []


In [13]:
# Step 9 - add month andyear columns for timeseries analysis
orders_delivered['order_month'] = orders_delivered['order_purchase_timestamp'].dt.to_period('M').astype(str)
orders_delivered['order_year'] = orders_delivered['order_purchase_timestamp'].dt.year
orders_delivered['order_quarter'] = orders_delivered['order_purchase_timestamp'].dt.to_period('Q').astype(str)

print(orders_delivered['order_month'].value_counts().sort_index().head(10))

order_month
2016-09       1
2016-10     265
2016-12       1
2017-01     750
2017-02    1653
2017-03    2546
2017-04    2303
2017-05    3546
2017-06    3135
2017-07    3872
Name: count, dtype: int64


### order_items table
1. Aggregated 112,650 rows to one row per order (96,478 rows) using groupby
2. Computed per-order aggregations:
   - total_items: count of items per order
   - total_revenue: sum of item prices
   - total_freight: sum of freight values
   - avg_item_price: mean item price
   - seller_id: first seller assigned to the order

In [14]:
# Step 1 - profile order_items
print(order_items.shape)
print(order_items.dtypes)
print(order_items.isnull().sum())
print(order_items.head())

(112650, 7)
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   
3  00024acbcdf0a6daa1e931b038114c75              1   
4  00042b26cf59d7ce69dfabb4e55b4fd9              1   

                         product_id                         seller_id  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   
1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36   
2  c777355d18b72b67abbeef

Observe that this is a relatively clean dataset as there are no nulls and the types look correct. One thing to note s that it has 112,650 rows but only 99,441 orders exist which suggest that some orders have multiple items. `order_item_id` tells us which item within a particular `order_id` it is (i.e. 1st/2nd...th order)

This is important to note for the join. If we join directly to `orders_delivered` we will get duplicate order rows for multi-item orders. We need to aggregate to order level first

In [15]:
# See how many orders have more than 1 item
item_counts = order_items.groupby('order_id')['order_item_id'].max()
print(item_counts.value_counts().head(10))

order_item_id
1     88863
2      7516
3      1322
4       505
5       204
6       198
7        22
8         8
10        8
12        5
Name: count, dtype: int64


In [16]:
# aggregate order_items to order level
order_items_agg = order_items.groupby('order_id').agg(
    total_items=('order_item_id', 'count'),
    total_revenue=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    avg_item_price=('price', 'mean'),
    seller_id=('seller_id', 'first')  # primary seller for the order
).reset_index()

print(order_items_agg.shape)
print(order_items_agg.head())

(98666, 6)
                           order_id  total_items  total_revenue  \
0  00010242fe8c5a6d1ba2dd792cb16214            1          58.90   
1  00018f77f2f0320c557190d7a144bdd3            1         239.90   
2  000229ec398224ef6ca0657da4fc703e            1         199.00   
3  00024acbcdf0a6daa1e931b038114c75            1          12.99   
4  00042b26cf59d7ce69dfabb4e55b4fd9            1         199.90   

   total_freight  avg_item_price                         seller_id  
0          13.29           58.90  48436dade18ac8b2bce089ec2a041202  
1          19.93          239.90  dd7ddc04e1b6c2c614352b383efe2d36  
2          17.87          199.00  5b51032eddd242adc84c38acab88f23d  
3          12.79           12.99  9d7a1d34a5052409006425275ba1c2b4  
4          18.14          199.90  df560393f3a51e74553ab94004ba5c87  


### customers table
- No cleaning required — already clean with no nulls
- Selected only customer_id and customer_state for the master join

In [17]:
# No cleaning needed for customers table
print(customers.shape)
print(customers.isnull().sum())
print(customers['customer_state'].value_counts().head(10))

(99441, 5)The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: int64


### products table
1. Merged with translation table on product_category_name to add English
   category names (LEFT JOIN — keeps all products)
2. Filled 623 untranslated category nulls with 'unknown'
3. Kept only product_id and product_category_name_english


In [29]:
# check nulls in products
print(products.isnull().sum())

# Merge portugese category names with english translation
products_translated = products.merge(
    translation,
    on='product_category_name',
    how='left'
)

# check how many products didn't get a translation
print("Untranslated:", products_translated['product_category_name_english'].isnull().sum())

# keep only what we need
products_clean = products_translated[['product_id', 'product_category_name_english']].copy()
print(products_clean.head())
print(products_clean.shape)

# fill nulls with 'unknown' before master join so they don't cause issues downstream
products_clean['product_category_name_english'] = (
    products_clean['product_category_name_english'].fillna('unknown')
)

print("Nulls remaining:", products_clean['product_category_name_english'].isnull().sum())

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
Untranslated: 623
                         product_id product_category_name_english
0  1e9e8ef04dbcff4541ed26657ea517e5                     perfumery
1  3aa071139cb16b67ca9e5dea641aaa2f                           art
2  96bd76ec8810374ed1b65e291975717f                sports_leisure
3  cef67bcfe19066a932b7673e239eb23d                          baby
4  9dc1a7de274444849c219cff195d0b71                    housewares
(32951, 2)
Nulls remaining: 0


### reviews table
1. Sorted by review_creation_date and deduplicated on order_id keeping
   the most recent review — removed 551 duplicate reviews
2. Kept only order_id and review_score

In [19]:
print(reviews.shape)
print(reviews.isnull().sum())

# some orders have multiple reviews - keep the most recent one
reviews_dedup = (reviews
    .sort_values('review_creation_date')
    .drop_duplicates('order_id', keep='last')
)

print(f"Before dedup: {len(reviews)}")
print(f"After dedup: {len(reviews_dedup)}")

# keep only what we need
reviews_clean = reviews_dedup[['order_id', 'review_score']].copy()

(99224, 7)
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64
Before dedup: 99224
After dedup: 98673


### Master join
Built olist_master.csv by left joining all cleaned tables to orders_delivered:
- orders_delivered (base) → order_items_agg → customers → reviews_clean
- Final shape: 96,478 rows × 25 columns
- Remaining nulls are expected and handled gracefully by Power BI/DAX

In [30]:
# start with orders_delivered as the base
master = orders_delivered.copy()

# join order items (revenue, freight, seller)
master = master.merge(order_items_agg, on='order_id', how='left')

# join customers (state only)
master = master.merge(
    customers[['customer_id', 'customer_state']],
    on='customer_id',
    how='left'
)

# join reviews (score only)
master = master.merge(reviews_clean, on='order_id', how='left')

print(f"Shape: {master.shape}")
print(master.isnull().sum())

Shape: (96478, 23)
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  14
order_delivered_carrier_date        2
order_delivered_customer_date       8
order_estimated_delivery_date       0
processing_days                    52
shipping_days                    1543
delivery_days                     314
delay_days                          8
is_late                             0
order_month                         0
order_year                          0
order_quarter                       0
total_items                         0
total_revenue                       0
total_freight                       0
avg_item_price                      0
seller_id                           0
customer_state                      0
review_score                      646
dtype: int64


The `products_clean` table is not included in the master table. Keep it as a separate table for category-level analysis in Power BI. This is cleaner as Power BI can handle the relationship between orders and prducts without flattening it into the master table.


### Exported clean files
- olist_master.csv — main analysis table (96,478 rows)
- olist_sellers.csv — seller dimension table
- olist_products.csv — product and category dimension table
- olist_order_items.csv — line item detail for product analysis

In [32]:
master.to_csv("data/clean/olist_master.csv", index=False)
print(f"Exported {len(master)} rows to data/clean/olist_master.csv")

# export sellers and product tables separately for dimension tables in Power BI
# sellers dimension
sellers.to_csv("data/clean/olist_sellers.csv", index=False)

# prpoducts with english category names
products_clean.to_csv("data/clean/olist_products.csv", index=False)

print("All clean files exported.")

Exported 96478 rows to data/clean/olist_master.csv
All clean files exported.


In [33]:
order_items.to_csv("data/clean/olist_order_items.csv", index=False)